# 02 - Osservazioni ISD e scelta dell'area

Secondo notebook del percorso didattico Nimbus. Qui si scarica l'inventario delle stazioni meteo
di superficie ISD (Integrated Surface Database, NOAA), si sceglie un'area geografica di lavoro e
si scaricano le osservazioni orarie di poche stazioni, per iniziare a maneggiare dati reali con i
loro limiti reali.

**Prerequisito:** il notebook `01-setup-e-fondamenti` deve essere gia' stato eseguito.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import common

common.richiede("00_env_report.json")
common.ensure_dirs()

# ---- SCEGLI QUI LA TUA AREA ----------------------------------------
# Puoi indicare una regione italiana ("lombardia", "sicilia", ...),
# una macroarea ("nord-ovest", "centro", "isole") oppure "italia".
REGIONE = common.REGIONE_DEFAULT
# --------------------------------------------------------------------

lon_min, lon_max, lat_min, lat_max = common.get_bbox(REGIONE)
print(f"Area scelta: {REGIONE}")
print(f"  longitudine {lon_min} -> {lon_max}")
print(f"  latitudine  {lat_min} -> {lat_max}")

## L'inventario delle stazioni

`isd-history.csv` e' l'inventario NOAA di tutte le stazioni ISD nel mondo: un elenco con
identificativi, coordinate, quota e **periodo di attivita'** (`BEGIN`/`END`). Non contiene alcuna
osservazione: nessuna temperatura, nessuna pressione, nessun dato misurato. E' solo l'anagrafica.

Scarichiamo questo file e riproduciamo i numeri gia' calcolati e riportati in
`docs/05-mvp-data-feasibility.md`, per verificare che il dato citato nel documento sia
effettivamente riproducibile da chiunque, invece di essere preso per fede.

In [ ]:
import pandas as pd

URL_INVENTARIO = "https://www.ncei.noaa.gov/pub/data/noaa/isd-history.csv"
percorso = common.scarica_con_cache(URL_INVENTARIO, common.RAW_DIR / "isd-history.csv")

inv = pd.read_csv(percorso, dtype=str)
print("Colonne:", list(inv.columns))
print("Record totali nel mondo:", len(inv))

it = inv[inv["CTRY"] == "IT"].copy()
it["lat"] = pd.to_numeric(it["LAT"], errors="coerce")
it["lon"] = pd.to_numeric(it["LON"], errors="coerce")
it["elev_m"] = pd.to_numeric(it["ELEV(M)"], errors="coerce")
it["begin"] = pd.to_datetime(it["BEGIN"], format="%Y%m%d", errors="coerce")
it["end"] = pd.to_datetime(it["END"], format="%Y%m%d", errors="coerce")

print("\nRecord con CTRY=IT:", len(it))          # atteso 318
geoloc = it[["lat", "lon"]].notna().all(axis=1).sum()
print("Con coordinate valorizzate:", geoloc)      # atteso 311

candidate = it[(it["begin"] <= "2021-01-01") & (it["end"] >= "2025-08-01")].copy()
print("Candidate 2021 -> ago 2025:", len(candidate))   # atteso 124
print("  sotto 300 m:", int((candidate["elev_m"] < 300).sum()))                                  # 89
print("  fra 300 e 999 m:", int(candidate["elev_m"].between(300, 999.999).sum()))                # 22
print("  da 1000 m:", int((candidate["elev_m"] >= 1000).sum()))                                  # 13

## I numeri e una discrepanza onesta

I valori 318, 124, 89/22/13 coincidono con la Misura 1 del documento di fattibilita': lo stesso
file, lo stesso filtro, lo stesso risultato. E' cosi' che si verifica un dato invece di fidarsi.

**Discrepanza da dichiarare:** il documento riporta 317 stazioni geolocalizzate, il conteggio
odierno ne trova 311. L'inventario e' aggiornato di continuo, quindi divergere di qualche unita' a
distanza di mesi e' atteso. La lezione: un numero misurato va sempre accompagnato dalla sua data.
Se la differenza fosse grande, andrebbe indagata prima di proseguire.

`END` non significa che la stazione sia spenta: molte righe si fermano ad agosto 2025 perche'
l'inventario e' aggiornato a blocchi.

In [ ]:
SOGLIA_MINIMA = 3

sel = candidate[
    candidate["lat"].between(lat_min, lat_max)
    & candidate["lon"].between(lon_min, lon_max)
].copy()
sel["station_id"] = sel["USAF"].str.strip() + "-" + sel["WBAN"].str.strip()
sel["nome"] = sel["STATION NAME"].str.strip()

print(f"Stazioni candidate dentro '{REGIONE}': {len(sel)}")
if len(sel):
    fasce = pd.cut(sel["elev_m"], [-100, 300, 1000, 9000],
                   labels=["<300 m", "300-999 m", ">=1000 m"], right=False)
    print(fasce.value_counts().sort_index().to_string())

if len(sel) < SOGLIA_MINIMA:
    print(f"\nTROPPO POCHE (minimo {SOGLIA_MINIMA}). Il percorso non prosegue con questo campione.")
    print("Scegli una macroarea piu' ampia e riesegui da questa cella:")
    print("  ", ", ".join(sorted(common.BBOX_MACROAREE)))
else:
    print("\nCampione sufficiente per proseguire.")

## Un bbox non e' un confine amministrativo

Il rettangolo geografico usato per filtrare le stazioni non conosce i confini regionali: prende
tutto cio' che cade dentro le sue coordinate, senza sapere se appartiene davvero alla regione
scelta. La mappa qui sotto serve a vedere cosa e' stato davvero preso, invece di fidarsi del nome
della variabile.

**Esempio concreto:** il bbox del Piemonte include anche Milano Linate, Malpensa, Cameri (che sono
in Lombardia) e Genova Sestri (che e' in Liguria). E' un limite accettato, reso visibile invece che
nascosto: filtrare un inventario di stazioni con un rettangolo non richiede un vero
point-in-polygon, ma bisogna sapere che l'approssimazione esiste.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    proiezione = ccrs.PlateCarree()
except Exception:
    ccrs = None

fig = plt.figure(figsize=(9, 9))
if ccrs is not None:
    ax = plt.axes(projection=proiezione)
    ax.set_extent(common.BBOX_ITALIA, crs=proiezione)
    ax.add_feature(cfeature.LAND, facecolor="#f2f2f2")
    ax.add_feature(cfeature.OCEAN, facecolor="#dceaf5")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4, linestyle=":")
    kw = {"transform": proiezione}
else:
    ax = plt.axes()
    lo0, lo1, la0, la1 = common.BBOX_ITALIA
    ax.set_xlim(lo0, lo1); ax.set_ylim(la0, la1); ax.set_aspect("equal")
    kw = {}

ax.scatter(candidate["lon"], candidate["lat"], s=14, c="#999999",
           label=f"candidate ISD ({len(candidate)})", **kw)
ax.add_patch(mpatches.Rectangle(
    (lon_min, lat_min), lon_max - lon_min, lat_max - lat_min,
    fill=False, edgecolor="#c0392b", linewidth=2, label=f"bbox {REGIONE}", **kw))
ax.scatter(sel["lon"], sel["lat"], s=70, c="#c0392b", edgecolor="white",
           zorder=5, label=f"selezionate ({len(sel)})", **kw)
for _, r in sel.iterrows():
    ax.annotate(f"{r['nome']}\n{r['elev_m']:.0f} m", (r["lon"], r["lat"]),
                fontsize=7, xytext=(4, 4), textcoords="offset points", **kw)

ax.set_title(f"Stazioni ISD candidate e area scelta: {REGIONE}")
ax.legend(loc="lower left", fontsize=8)
plt.show()

## Scrittura dell'inventario selezionato

Le stazioni selezionate dentro l'area scelta vengono salvate in `01_stations.csv`: e' l'artefatto
che il resto del percorso didattico usera' come elenco di stazioni di riferimento.

In [ ]:
colonne = ["station_id", "usaf", "wban", "nome", "lat", "lon", "elev_m", "begin", "end"]
out = sel.rename(columns={"USAF": "usaf", "WBAN": "wban"})[colonne].copy()
out["regione_scelta"] = REGIONE
out.to_csv(common.data_path("01_stations.csv"), index=False)
print("Scritte", len(out), "stazioni in", common.data_path("01_stations.csv"))

## Poche stazioni, pochi mesi

Si scaricano soltanto le osservazioni di un anno e di poche stazioni perche' il percorso deve
restare eseguibile per intero in una sessione di lavoro, senza scaricare gigabyte di archivio
pluriennale. Il comando per l'archivio storico completo esiste ed e' mostrato qui sotto solo come
riferimento, ma non viene eseguito in questo notebook:

```text
# Solo come riferimento, NON eseguito:
# https://www.ncei.noaa.gov/pub/data/noaa/{anno}/{USAF}-{WBAN}-{anno}.gz
# per ogni anno dal primo al ultimo anno di interesse.
```

Le stazioni vengono scelte tra quelle appena selezionate, ordinandole per quota decrescente e
prendendone al massimo `MAX_STAZIONI`, cosi' da avere un piccolo campione variegato in altitudine.

In [ ]:
ANNO = 2024
MAX_STAZIONI = 5
scelte = out.sort_values("elev_m", ascending=False).head(MAX_STAZIONI)

frames = []
for _, r in scelte.iterrows():
    nome_file = f"{r['usaf']}-{r['wban']}-{ANNO}.gz"
    url = f"https://www.ncei.noaa.gov/pub/data/noaa/{ANNO}/{nome_file}"
    try:
        p = common.scarica_con_cache(url, common.RAW_DIR / nome_file)
    except Exception as e:
        print(f"  {r['nome']}: non disponibile ({e})")
        continue
    # ISD lite/full e' a larghezza fissa: qui leggiamo i campi obbligatori
    # posizionali del record (data, ora, temperatura e relativo flag QA).
    righe = []
    import gzip
    with gzip.open(p, "rt", errors="ignore") as f:
        for linea in f:
            try:
                data = linea[15:23]          # AAAAMMGG
                ora = linea[23:27]           # HHMM
                t_raw = linea[87:92]         # temperatura in decimi di C
                t_qa = linea[92]             # flag qualita'
                if t_raw == "+9999":
                    continue
                righe.append((f"{data}{ora}", int(t_raw) / 10.0, t_qa))
            except (ValueError, IndexError):
                continue
    if not righe:
        print(f"  {r['nome']}: nessuna temperatura utilizzabile")
        continue
    df = pd.DataFrame(righe, columns=["stamp", "t2m_c", "qa_flag"])
    df["valid_time_utc"] = pd.to_datetime(df["stamp"], format="%Y%m%d%H%M", utc=True)
    df["station_id"] = r["station_id"]
    frames.append(df[["station_id", "valid_time_utc", "t2m_c", "qa_flag"]])
    print(f"  {r['nome']}: {len(df)} osservazioni")

oss = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(
    columns=["station_id", "valid_time_utc", "t2m_c", "qa_flag"])

## Dati sporchi: QA e completezza

`+9999` e' il sentinella ISD per "temperatura assente", non una temperatura: e' gia' stato escluso
nella cella precedente. Il flag QA (`qa_flag`) distingue valori validati da valori sospetti, ma la
validazione e' automatica: non certifica che il dato sia corretto, solo che ha superato controlli di
plausibilita' di base.

I buchi nella serie oraria esistono e non vanno riempiti in silenzio: la cella seguente li rende
visibili, stazione per stazione, invece di nasconderli dietro un'interpolazione.

In [ ]:
print("Osservazioni totali:", len(oss))
print("\nDistribuzione dei flag QA:")
print(oss["qa_flag"].value_counts().to_string())

# I flag "2", "3", "6", "7" indicano valori sospetti o errati nel formato ISD.
sospetti = oss["qa_flag"].isin(list("2367"))
print(f"\nValori marcati sospetti: {int(sospetti.sum())} su {len(oss)}")

print("\nCompletezza oraria per stazione:")
for sid, g in oss.groupby("station_id"):
    attese = pd.date_range(g["valid_time_utc"].min(), g["valid_time_utc"].max(),
                           freq="h", tz="UTC")
    presenti = g["valid_time_utc"].dt.floor("h").nunique()
    print(f"  {sid}: {presenti}/{len(attese)} ore = {100*presenti/len(attese):.1f}%")

oss.to_parquet(common.data_path("02_observations.parquet"), index=False)
print("\nScritto", common.data_path("02_observations.parquet"))

## Il limite di quello che hai fatto

- **Completezza della stazione != completezza della variabile**: l'inventario dice che la stazione
  era attiva, non che la temperatura ci sia ogni ora. Il conteggio appena stampato lo dimostra.
- Il campione e' di poche stazioni e un anno: sufficiente per imparare, insufficiente per
  concludere.
- Il flag QA di ISD e' automatico. Come ricorda `docs/03-italy-observation-source-census.md`,
  validazione automatica non equivale a validazione finale.
- Il bbox include stazioni fuori regione: le hai viste sulla mappa.